## Benchmark with other methods

Categories:
- (Multi-omics) latent representation
- Gradient computation & Trajectory inference

In [ ]:
import os
import gc
import sys

import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

import torch
import torch.nn as nn
import tifffile
import seaborn as sns
import matplotlib.pyplot as plt

from torch_geometric import utils as pyg_utils
from scipy.stats import spearmanr
from sklearn.metrics import normalized_mutual_info_score

In [ ]:
from ipywidgets import interact, widgets
from IPython.display import display

from matplotlib import rcParams
rcParams['font.family'] = 'FreeSans'
rcParams.update({'font.size': 10})
rcParams.update({'figure.dpi': 300})
rcParams.update({'savefig.dpi': 300})

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
sys.path.append('..')
sys.path.append('../models/')
sys.path.append('../util')

import IO, utils, plot, trajectory, metrics
import vgae
import scFates as scf

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Utils
from sklearn.metrics import roc_auc_score, f1_score

def compute_spearman(zonation, adata_antibody):
    factors = np.unique(zonation)
    corrs = np.zeros((len(factors), len(adata_antibody.var_names)))
    
    for i, factor in enumerate(factors):
        for j, label in enumerate(adata_antibody.var_names):
            corrs[i, j] = spearmanr(
                (zonation == factor).astype(np.uint8).values,
                adata_antibody[:, label].X.squeeze()   
            )[0]
            
    return corrs


def compute_auroc(t, adata_antibody, threshold=0.5):
    auroc = np.zeros_like(adata_antibody.var_names)
    for i, marker in enumerate(adata_antibody.var_names):
        target = adata_antibody[:, marker].X.squeeze() > threshold
        if 'GS' in marker or 'CYP' in marker:
            try:
                auroc[i] = roc_auc_score(target, t)
            except ValueError:
                auroc[i] = 0
        else:
            try:
                auroc[i] = roc_auc_score(target, 1-t)
            except ValueError:
                auroc[i] = 0

    return auroc


def kendall_tau_distance(u, v):
    """
    Compute normalized Kendall Tau distance btw 2 (permutated) rankings
    Reference: https://en.wikipedia.org/wiki/Kendall_tau_distance
    """
    n = len(u)
    assert len(v) == n, "Both lists have to be of equal length"
    i, j = np.meshgrid(np.arange(n), np.arange(n))
    a = np.argsort(u)
    b = np.argsort(v)
    ndisordered = np.logical_or(np.logical_and(a[i] < a[j], b[i] > b[j]), np.logical_and(a[i] > a[j], b[i] < b[j])).sum()
    return ndisordered / (n * (n - 1))


Load sample data: <br>

In [ ]:
xenium_path = '../data/xenium/'
desi_path = '../data/desi/'
ab_path = '../data/integrated/antibody/'

sample_id = 'NIH_F5'

adata_xenium = IO.load_xenium(os.path.join(xenium_path, sample_id))
adata_desi = sc.read_h5ad(os.path.join(desi_path, sample_id+'.h5'))
adata_xenium, adata_desi = IO.filter_cells(adata_xenium, adata_desi, by='map')

# Antibody (validation)
adata_ab = IO.load_ab_stain(
    os.path.join(ab_path, sample_id+'.ome.tif'),
    adata_ref=adata_xenium
)

# Normalize to [0, 1] per channel
scaled_chans = np.zeros_like(adata_ab.X)
for i, chan in enumerate(adata_ab.X.T):
    chan = (chan-chan.min()) / (chan.max()-chan.min())
    scaled_chans[:, i] = chan

adata_ab.X = scaled_chans

ab_dict = {
    'Opal 690-GS': 'Central Vein',
    'Opal 780-CYP3A4': 'Peri-central',
    'Opal 570-ASS1': 'Peri-portal',
    'Opal 520-Col1': 'Portal Vein'
}
ab_labels = list(ab_dict.keys())

### Ablation

In [ ]:
n_latent = 6

In [ ]:
from scipy.spatial import cKDTree

def correct_mislabel_veins(adata, use_rep='ab_label', k=10):
    spatial_coords = adata.obsm["spatial"]
    labels = adata.obs[use_rep].values

    # Get indices for each label
    idx_label_1 = np.where(labels == 1)[0]
    idx_label_2 = np.where(labels == 2)[0]
    idx_label_3 = np.where(labels == 3)[0]

    # Build KD-trees for fast nearest-neighbor search
    tree_label_1 = cKDTree(spatial_coords[idx_label_1])
    tree_label_2 = cKDTree(spatial_coords[idx_label_2])

    # Find average distances from each label 3 cell to label 1 and label 2
    d1, _ = tree_label_1.query(spatial_coords[idx_label_3], k=k, workers=-1)
    d2, _ = tree_label_2.query(spatial_coords[idx_label_3], k=k, workers=-1)

    avg_d1 = d1.mean(axis=1)
    avg_d2 = d2.mean(axis=1)

    # Identify mislabeled 3s where avg distance to 1 is smaller than to 2
    mislabeled = avg_d1 < avg_d2
    labels[idx_label_3[mislabeled]] = 0  # Correct misclassified labels

    # Denoise
    labels = adata.obs[use_rep].values

    # Get indices for labels 0 & 3
    idx_label_0 = np.where(labels == 0)[0]
    idx_label_3 = np.where(labels == 3)[0]
    idx_0_3 = np.concatenate([idx_label_0, idx_label_3])  # Only process 0 & 3

    # Build KD-tree for spatial queries
    tree = cKDTree(spatial_coords)

    # Query nearest neighbors (excluding self)
    _, neighbors = tree.query(spatial_coords[idx_0_3], k=k+1, workers=-1)  # k+1 to exclude self

    # Count majority labels in neighbors
    for i, idx in enumerate(idx_0_3):
        neighbor_labels = labels[neighbors[i, 1:]]  # Exclude self
        majority_label = np.bincount(neighbor_labels).argmax()  # Most frequent label

        # Only update if the majority is different from the current label
        if majority_label in {0, 3} and majority_label != labels[idx]:
            labels[idx] = majority_label

    # Update AnnData object
    adata.obs[use_rep] = labels
    adata.obs[use_rep] = adata.obs[use_rep].astype('category')
    return None



In [ ]:
def calculate_vein_axis(
    adata, use_rep='ab_label', 
    w1=.5, w2=.5, 
    vmin=0., vmax=1., k=10
):
    r"""Approx. annotation of PV -> CV axis from only antibody imaging"""
    # Extract spatial coordinates
    coords = adata.obsm['spatial']

    # Identify indices for each structure
    cv_indices = np.where(adata.obs[use_rep] == 0)[0]
    pp_indices = np.where(adata.obs[use_rep] == 2)[0]
    pv_indices = np.where(adata.obs[use_rep] == 3)[0]

    # Build KD-Trees for each structure
    cv_tree = cKDTree(coords[cv_indices])
    pp_tree = cKDTree(coords[pp_indices])
    pv_tree = cKDTree(coords[pv_indices])

    # Initialize array to hold PV-CV values
    axis_values = np.zeros(coords.shape[0])

    # Calculate mean distances to each structure
    for i, point in enumerate(coords):
        d_cv, _ = cv_tree.query(point, k=k)
        d_pp, _ = pp_tree.query(point, k=k)
        d_pv, _ = pv_tree.query(point, k=k)

        mu_cv = np.mean(d_cv)
        mu_pp = np.mean(d_pp)
        mu_pv = np.mean(d_pv)

        # Calculate H1(Medulla vs. Cortex) & H2 (Cortex vs. Capsule)
        H1 = (mu_pp - mu_pv) / (mu_pp + mu_pv)
        H2 = (mu_cv - mu_pp) / (mu_cv + mu_pp)

        # Combine H1 and H2 to get CMA value
        axis_values[i] = w1 * H1 + w2 * H2

    # Rescale to [vmin, vmax]
    axis_min, axis_max = np.min(axis_values), np.max(axis_values)
    axis_values = vmin + (axis_values - axis_min) * ((vmax-vmin) / (axis_max-axis_min))

    adata.obs['t'] = 1.0 - axis_values   # PV (0) --> CV (1)
    return None

In [ ]:
# Obtain 1-hot encoded argmax
argmax_expr = adata_ab.X.argmax(1)
adata_ab.obs['ab_label'] = argmax_expr
correct_mislabel_veins(adata_ab, k=50)
calculate_vein_axis(adata_ab, k=10)


In [ ]:
sq.pl.spatial_scatter(
    adata_ab, color='ab_label', groups=[0, 3], img=False, size=30
)

In [ ]:
sq.pl.spatial_scatter(
    adata_ab, color='t', img=False, size=30, cmap='RdBu_r'
)

In [ ]:
# Save "silver-standard" annotation
np.save('../results/liver/ablation/antibody_gamma.npy', adata_ab.obs['t'].values)

In [ ]:
# Load LYNX v & zs
lynx_df = pd.read_csv('../results/liver/ablation/lynx.csv', index_col=[0])
lynx_no_c_df = pd.read_csv('../results/liver/ablation/lynx_no_c.csv', index_col=[0])
lynx_no_u_df = pd.read_csv('../results/liver/ablation/lynx_no_u.csv', index_col=[0])
vgae_df = pd.read_csv('../results/liver/ablation/VGAE.csv', index_col=[0])


In [ ]:
adata_xenium.obs['t_lynx'] = lynx_df['t'].values
adata_xenium.obs['t_lynx_no_c'] = lynx_no_c_df['t'].values
adata_xenium.obs['t_lynx_no_u'] = lynx_no_u_df['t'].values
adata_xenium.obs['t_vgae'] = vgae_df['t'].values

### PCA, Diffusion map, DPT

In [ ]:
sc.pp.normalize_total(adata_xenium)
sc.pp.log1p(adata_xenium)
sc.pp.pca(adata_xenium)
sc.pp.neighbors(adata_xenium, n_neighbors=50)
sc.tl.diffmap(adata_xenium, n_comps=10)

In [ ]:
sq.pl.spatial_scatter(adata_xenium, color='DPT', img=False, size=30, cmap='Reds')

Use PC0, DC1, rotate with `DPT` gene expression direction

In [ ]:
# Use PC1, DC1, rotate with `DPT` gene expression direction
pc1 = adata_xenium.obsm['X_pca'][:, 1]
pc1 = (pc1 - pc1.min()) / (pc1.max() - pc1.min())
adata_xenium.obs['t_pca'] = pc1
sq.pl.spatial_scatter(adata_xenium, color='t_pca', img=False, size=20, cmap='RdBu_r')


In [ ]:
dc1 = adata_xenium.obsm['X_diffmap'][:, 1]
dc1 = (dc1 - dc1.min()) / (dc1.max() - dc1.min())
adata_xenium.obs['t_diffmap'] = 1. - dc1
sq.pl.spatial_scatter(adata_xenium, color='t_diffmap', img=False, size=20, cmap='RdBu_r')

In [ ]:
adata_xenium.uns['iroot'] = adata_xenium[:, 'DPT'].X.A.argmax()
sc.tl.dpt(adata_xenium)

t_dpt = adata_xenium.obs['dpt_pseudotime'].copy()
t_dpt = (t_dpt-t_dpt.min()) / (t_dpt.max()-t_dpt.min())
adata_xenium.obs['t_dpt'] = t_dpt
adata_xenium.obs.drop('dpt_pseudotime', inplace=True, axis=1)
sq.pl.spatial_scatter(adata_xenium, color='t_dpt', img=False, size=20, cmap='RdBu_r')

### CORAL

In [ ]:
t_coral = pd.read_csv('../results/liver/Coral_pseudotime.csv', index_col=[0])['t'].values
adata_xenium.obs['t_coral'] = t_coral

In [ ]:
sq.pl.spatial_scatter(
    adata_xenium, color='t_coral', 
    cmap='RdBu_r', size=20, img=False,
    title=r'Spatial Trajectory ($\gamma(t)$)'+'\nCORAL (Xenium)'
)


### GASTON

In [ ]:
adata_xenium_gaston = sc.read_h5ad('../results/liver/adata_xenium_gaston_dim20.h5')
adata_xenium_gaston = adata_xenium_gaston[adata_xenium.obs_names]
adata_xenium.obs['t_gaston'] = adata_xenium_gaston.obs['isodepth'].values.copy()


In [ ]:
sq.pl.spatial_scatter(
    adata_xenium_gaston, color='isodepth', 
    cmap='RdBu_r', size=20, img=False,
    title=r'Spatial Trajectory ($\gamma(t)$)'+'\nGASTON(Xenium)',
    vmin=0.25, vmax=0.65
)

In [ ]:
auroc_gaston = np.array([
    compute_auroc(adata_xenium_gaston.obs['isodepth'].values, adata_ab, threshold)
    for threshold in np.linspace(0.1, 0.9, 9)
])

### scVI / MultiVI

In [ ]:
import scvi
scvi.settings.seed = 42
print("Last run with scvi-tools version:", scvi.__version__)

#### scVI

Model training:

In [ ]:
scvi.model.SCVI.setup_anndata(adata_xenium)

In [ ]:
n_latent = 6
model = scvi.model.SCVI(
    adata_xenium, 
    n_layers=2, 
    n_latent=n_latent, 
    gene_likelihood="nb"
)

model.train()

In [ ]:
latent = model.get_latent_representation()
np.save('../results/liver/scvi_{}.npy'.format(n_latent), latent)
adata_xenium.obsm['X_scvi'] = latent

Trajectory inference:

In [ ]:
adata_xenium.obsm['X_scvi'] = np.load('../results/liver/scvi_{}.npy'.format(n_latent))
trajectory.compute_trajectory(
    adata_xenium, 
    use_rep='X_scvi',
    root_marker='DPT'
)

adata_xenium.obs['t_scvi'] = adata_xenium.obs['t'].copy()
adata_xenium.obs.drop('t', inplace=True, axis=1)


In [ ]:
# Spatial visualization
sq.pl.spatial_scatter(
    adata_xenium, color='t', 
    cmap='RdBu_r', size=20, img=False,
    title=r'Spatial Trajectory ($\gamma(t)$)'+'\nscVI (Xenium)',
)

Antibody validation:

Spatial heterogeneity metrics:

- (1). Moran's I, Geary's C (Ripley: measure cell density instead)
- (2). Network's centrality
  - degree centrality: % non-group members connected to group members
  - closeness centrality: closeness the group to other nodes

In [ ]:
# Spatial heterogeneity metrics
sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)
sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='moran', transformation=False
)
sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='geary', transformation=False
)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_scvi = adata.uns['moranI'].loc['t', 'I']
geary_scvi = adata.uns['gearyC'].loc['t', 'C']
centrality_scvi = adata.uns['milestones_centrality_scores']['degree_centrality'].values


#### TotalVI / MultiVI

In [ ]:
import anndata as ad

In [ ]:
# Concatenate `var` for joint Xenium & DESI
n_genes = adata.shape[1]
n_metabolites = adata_desi.shape[1]

adata_mvi = ad.concat((adata, adata_desi), axis=1, join='inner')
adata_mvi.var['modality'] = ['Gene Expression']*n_genes + ['DESI Intensity']*n_metabolites
adata_mvi.obsm['spatial'] = adata.obsm['spatial'].copy()
adata_mvi.uns['spatial'] = adata.uns['spatial'].copy()

In [ ]:
scvi.model.MULTIVI.setup_anndata(adata_mvi)

In [ ]:
n_latent = 8
n_hidden = 16

model = scvi.model.MULTIVI(
    adata_mvi,
    n_genes=n_genes,
    n_regions=n_metabolites,
    n_hidden=n_hidden,
    n_latent=n_latent,
    gene_likelihood="nb"
)

In [ ]:
model.train(adversarial_mixing=False)

In [ ]:
latent = model.get_latent_representation()
adata.obsm['X_multivi'] = latent
np.save('../results/multivi_{}.npy'.format(n_latent), latent)

Trajectory inference:

In [ ]:
n_latent = 8
latent = np.load('../results/multivi_{}.npy'.format(n_latent))
adata.obsm['X_multivi'] = latent

In [ ]:
trajectory.compute_trajectory(
    adata, 
    use_rep='X_multivi',
    dist_metric='euclidean'
)

# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

t_multivi = adata.obs['t']
zone_multivi = adata.obs['milestones']

In [ ]:
# UMAP visualization
sc.pp.neighbors(adata, use_rep='X_multivi')
sc.tl.umap(adata)
scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent))) 

In [ ]:
# Spatial visualization
sq.pl.spatial_scatter(
    adata, color='t', 
    vmin=np.quantile(adata.obs['t'], .05),
    vmax=np.quantile(adata.obs['t'], .95),
    cmap='RdBu_r', size=20, img=False,
    title='Spatial Gradients\n (MultiVI)'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab10', size=20, img=False,
    title='Zonation\n (MultiVI)'
)

MultiVI:

In [ ]:
# AUROC score against thresholded antibody channel
auroc_multivi = np.array([
    compute_auroc(t_multivi, adata_ab, threshold)
    for threshold in np.linspace(0.1, 0.9, 9)
])

In [ ]:
# Spatial heterogeneity metrics
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='moran', transformation=False
)
sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='geary', transformation=False
)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_multivi = adata.uns['moranI'].loc['t', 'I']
geary_multivi = adata.uns['gearyC'].loc['t', 'C']
centrality_multivi = adata.uns['milestones_centrality_scores']['degree_centrality'].values


### CLUE

In [ ]:
import anndata as ad
import scglue
import networkx as nx

In [ ]:
# Configure model
scglue.models.configure_dataset(
    adata, "NB", use_highly_variable=False
)

scglue.models.configure_dataset(
    adata_desi, "Normal", use_highly_variable=False
)

In [ ]:
# First try without pretraining
n_latent = 8
n_hidden = 16

model = scglue.models.SCCLUEModel(
    adatas={"rna": adata, "metabolite": adata_desi},
    latent_dim=n_latent,
    x2u_h_dim=n_hidden,
    u2x_h_dim=n_hidden,
    random_seed=42
)

model.compile()
model.fit(adatas={"rna": adata, "metabolite": adata_desi})

In [ ]:
# DEBUG: why `n_latent` doubled?? multi-modal concatenation within `model.encode_data`? 
adata.obsm['X_glue'] = model.encode_data("rna", adata)
adata_desi.obsm['X_glue'] = model.encode_data("metabolite", adata_desi)
np.save('../results/clue_{}.npy'.format(n_latent), adata.obsm['X_glue'])

Trajectory inference:

In [ ]:
n_latent = 8 
latent = np.load('../results/clue_{}.npy'.format(n_latent))
adata.obsm['X_glue'] = latent

In [ ]:
# Debug: why `n_latent` doubled?: Confirm w/ author: concatenation of q(z1 | x1) || q(z1 | x2)
n_nodes = 8

trajectory.compute_trajectory(
    adata, 
    use_rep='X_glue',
    ndim=n_latent*2,
    dist_metric='euclidean', 
)

# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

t_clue = adata.obs['t']
zone_clue = adata.obs['milestones']

In [ ]:
# UMAP visualization
sc.pp.neighbors(adata, use_rep='X_glue')
sc.tl.umap(adata)
scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_nodes))) 

In [ ]:
# Spatial visualization
sq.pl.spatial_scatter(
    adata, color='t', 
    vmin=np.quantile(adata.obs['t'], .05),
    vmax=np.quantile(adata.obs['t'], .95),
    cmap='RdBu_r', size=20, img=False,
    title='Spatial Gradients\n (CLUE)'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab10', size=20, img=False,
    title='Zonation\n (CLUE)'
)

Antibody validation:

In [ ]:
# AUROC score against thresholded antibody channel
auroc_clue = np.array([
    compute_auroc(t_clue, adata_ab, threshold)
    for threshold in np.linspace(0.1, 0.9, 9)
])

In [ ]:
# Spatial heterogeneity metrics
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='moran', transformation=False
)
sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='geary', transformation=False
)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_clue = adata.uns['moranI'].loc['t', 'I']
geary_clue = adata.uns['gearyC'].loc['t', 'C']
centrality_clue = adata.uns['milestones_centrality_scores']['degree_centrality'].values


### MOFA+

In [ ]:
import muon as mu

In [ ]:
adata.var['highly_variable'] = True
adata_desi.var['highly_variable'] = True

mdata = mu.MuData({"rna": adata, "metabolite": adata_desi})
mdata.var['highly_variable']=mdata.var['highly_variable'].to_list()
mdata

In [ ]:
n_latent = 8
mu.tl.mofa(mdata, n_factors=n_latent, seed=42)

In [ ]:
np.save('../results/mofa_{}.npy'.format(n_latent), mdata.obsm['X_mofa'])

Trajectory Inference:

In [ ]:
n_latent = 8 
latent = np.load('../results/mofa_{}.npy'.format(n_latent))
adata.obsm['X_mofa'] = latent

In [ ]:
trajectory.compute_trajectory(
    adata, 
    use_rep='X_mofa',
    dist_metric='euclidean', 
)

# Optional: rotate trajectory direction by 180-deg
adata.obs['t'] = adata.obs['t'].max() - adata.obs['t']
adata.obs['milestones'] = adata.obs['milestones'].astype(int).max() - adata.obs['milestones'].astype(int)
adata.obs['milestones'] = adata.obs['milestones'].astype('category')

t_mofa = adata.obs['t']
zone_mofa = adata.obs['milestones']

In [ ]:
# UMAP visualization
sc.pp.neighbors(adata, use_rep='X_mofa')
sc.tl.umap(adata)
scf.pl.graph(adata, basis='umap', nodes=list(np.arange(n_latent))) 

In [ ]:
# Spatial visualization
sq.pl.spatial_scatter(
    adata, color='t', 
    vmin=np.quantile(adata.obs['t'], .03),
    vmax=np.quantile(adata.obs['t'], .97),
    cmap='RdBu_r', size=20, img=False,
    colorbar=None,
    title='Spatial Gradients\n (MOFA+)',
    # save='../results/plot/mofa.pdf'
)

sq.pl.spatial_scatter(
    adata, color='milestones', 
    cmap='tab10', size=20, img=False,
    title='Zonation\n (MOFA+)'
)

In [ ]:
# AUROC score against thresholded antibody channel
auroc_mofa = np.array([
    compute_auroc(t_mofa, adata_ab, threshold)
    for threshold in np.linspace(0.1, 0.9, 9)
])

In [ ]:
# Spatial heterogeneity metrics
# sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='moran', transformation=False
)
sq.gr.spatial_autocorr(
    adata, attr='obs', genes=['t'],
    mode='geary', transformation=False
)
sq.gr.centrality_scores(adata, cluster_key='milestones')

display(adata.uns['moranI'])
display(adata.uns['gearyC'])
display(adata.uns['milestones_centrality_scores'])

moran_mofa = adata.uns['moranI'].loc['t', 'I']
geary_mofa = adata.uns['gearyC'].loc['t', 'C']
centrality_mofa = adata.uns['milestones_centrality_scores']['degree_centrality'].values


### Metrics summary

**Continuous trajectory vs. antibody annotation**

In [ ]:
# Benchmark among ablations
rand_indices = np.random.choice(
    np.arange(adata_xenium.shape[0]), 10000, replace=False
)
plot.disp_kde_scatter(
    adata_xenium.obs['t_lynx'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
    xlabel=r"Antibody-annotated $\gamma(t)$",
    ylabel=r"LYNX prediction $\gamma(t)$",
    title="Spatial Trajectory\n LYNX vs. Antibody"
)

plot.disp_kde_scatter(
    adata_xenium.obs['t_lynx_no_c'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
    xlabel=r"Antibody-annotated $\gamma(t)$",
    ylabel=r"LYNX (no c) prediction $\gamma(t)$",
    title="Spatial Trajectory\n LYNX (no c) vs. Antibody"
)

plot.disp_kde_scatter(
    adata_xenium.obs['t_lynx_no_u'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
    xlabel=r"Antibody-annotated $\gamma(t)$",
    ylabel=r"LYNX (no_u) prediction $\gamma(t)$",
    title="Spatial Trajectory\n LYNX (no u) vs. Antibody"
)

plot.disp_kde_scatter(
    adata_xenium.obs['t_vgae'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
    xlabel=r"Antibody-annotated $\gamma(t)$",
    ylabel=r"Xenium VGAE prediction $\gamma(t)$",
    title="Spatial Trajectory\n Xenium VGAE vs. Antibody"
)

plot.disp_kde_scatter(
    adata_xenium.obs['t_coral'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
    xlabel=r"Antibody-annotated $\gamma(t)$",
    ylabel=r"CORAL prediction $\gamma(t)$",
    title="Spatial Trajectory\n CORAL vs. Antibody"
)

plot.disp_kde_scatter(
    adata_xenium.obs['t_gaston'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
    xlabel=r"Antibody-annotated $\gamma(t)$",
    ylabel=r"GASTON prediction $\gamma(t)$",
    title="Spatial Trajectory\n GASTON vs. Antibody"
)

plot.disp_kde_scatter(
    adata_xenium.obs['t_scvi'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
    xlabel=r"Antibody-annotated $\gamma(t)$",
    ylabel=r"scVI prediction $\gamma(t)$",
    title="Spatial Trajectory\n scVI vs. Antibody"
)

plot.disp_kde_scatter(
    adata_xenium.obs['t_pca'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
    xlabel=r"Antibody-annotated $\gamma(t)$",
    ylabel=r"PCA prediction $\gamma(t)$",
    title="Spatial Trajectory\n PCA vs. Antibody"
)

plot.disp_kde_scatter(
    adata_xenium.obs['t_diffmap'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
    xlabel=r"Antibody-annotated $\gamma(t)$",
    ylabel=r"Diffmap prediction $\gamma(t)$",
    title="Spatial Trajectory\nDiffmap vs. Antibody"
)

plot.disp_kde_scatter(
    adata_xenium.obs['t_dpt'].values[rand_indices], adata_ab.obs['t'].values[rand_indices],
    xlabel=r"Antibody-annotated $\gamma(t)$",
    ylabel=r"DPT prediction $\gamma(t)$",
    title="Spatial Trajectory\nDPT vs. Antibody"
)


In [ ]:
%reload_ext autoreload

**AUROC scores** (against individual antibody channels)

In [ ]:
ab_channels = ab_labels[::-1]
ab_channels

In [ ]:
# Try ROC curve instead??

from sklearn.metrics import RocCurveDisplay, roc_curve, auc

In [ ]:
methods = ['LYNX', 'PCA', 'Diffmap', 'DPT', 'scVI', 'CORAL', 'GASTON']
ts = ['t_lynx', 't_pca', 't_diffmap', 't_dpt', 't_scvi', 't_coral', 't_gaston']
custom_palette = [
    "#d400ff",   # Bright Purple 
    "#003f5c",  # Deep Blue
    "#ffa600",  # Bright Orange
    "#bc5090",  # Vivid Magenta
    "#ff6361",  # Bright Red
    "#9e3b1e",  # Coffee?
    "#28a745",  # Vivid Green
]

In [ ]:
y_gs = (adata_ab[:, ab_labels[0]].X > 0.8).squeeze().astype(np.uint8)
y_cyp = (adata_ab[:, ab_labels[1]].X > 0.6).squeeze().astype(np.uint8)
y_ass = (adata_ab[:, ab_labels[2]].X < 0.4).squeeze().astype(np.uint8)
y_col1 = (adata_ab[:, ab_labels[3]].X < 0.2).squeeze().astype(np.uint8)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7), dpi=300)
for i in range(len(methods)):
    y_pred = adata_xenium.obs[ts[i]].values
    method = methods[i]

    fpr, tpr, _ = roc_curve(y_gs, y_pred)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{method} (AUC = {roc_auc:.2f})", lw=2, c=custom_palette[i])

ax.plot([0, 1], [0, 1], 'k--', label="Random (AUC = 0.5)")
ax.set_xlabel('False Positive Rate', fontsize=20)
ax.set_ylabel('True Positive Rate', fontsize=20)
ax.spines[['right', 'top']].set_visible(False)
leg = ax.legend(bbox_to_anchor=(1.01, 0.8), loc='upper left', borderaxespad=0, fontsize=20)
for line in leg.get_lines():
    line.set_linewidth(5)
ax.set_title('ROC Curves against Antibody\n(GS: central vein)', fontsize=20)
fig.show()


fig, ax = plt.subplots(figsize=(8, 7), dpi=300)
for i in range(len(methods)):
    y_pred = adata_xenium.obs[ts[i]].values
    method = methods[i]

    fpr, tpr, _ = roc_curve(y_cyp, y_pred)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{method} (AUC = {roc_auc:.2f})", lw=2, c=custom_palette[i])

ax.plot([0, 1], [0, 1], 'k--', label="Random (AUC = 0.5)")
ax.set_xlabel('False Positive Rate', fontsize=20)
ax.set_ylabel('True Positive Rate', fontsize=20)
ax.spines[['right', 'top']].set_visible(False)
leg = ax.legend(bbox_to_anchor=(1.01, 0.8), loc='upper left', borderaxespad=0, fontsize=20)
for line in leg.get_lines():
    line.set_linewidth(5)
ax.set_title('ROC Curves against Antibody\n(CYP3A4: peri-central)', fontsize=20)
fig.show()


fig, ax = plt.subplots(figsize=(8, 7), dpi=300)
for i in range(len(methods)):
    y_pred = adata_xenium.obs[ts[i]].values
    method = methods[i]

    fpr, tpr, _ = roc_curve(y_ass, y_pred)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{method} (AUC = {roc_auc:.2f})", lw=2, c=custom_palette[i])

ax.plot([0, 1], [0, 1], 'k--', label="Random (AUC = 0.5)")
ax.set_xlabel('False Positive Rate', fontsize=20)
ax.set_ylabel('True Positive Rate', fontsize=20)
ax.spines[['right', 'top']].set_visible(False)
leg = ax.legend(bbox_to_anchor=(1.01, 0.8), loc='upper left', borderaxespad=0, fontsize=20)
for line in leg.get_lines():
    line.set_linewidth(5)
ax.set_title('ROC Curves against Antibody\n(ASS1: peri-portal)', fontsize=20)
fig.show()


fig, ax = plt.subplots(figsize=(8, 7), dpi=300)
for i in range(len(methods)):
    y_pred = adata_xenium.obs[ts[i]].values
    method = methods[i]

    fpr, tpr, _ = roc_curve(y_col1, y_pred)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{method} (AUC = {roc_auc:.2f})", lw=2, c=custom_palette[i])

ax.plot([0, 1], [0, 1], 'k--', label="Random (AUC = 0.5)")
ax.set_xlabel('False Positive Rate', fontsize=20)
ax.set_ylabel('True Positive Rate', fontsize=20)
ax.spines[['right', 'top']].set_visible(False)
leg = ax.legend(bbox_to_anchor=(1.01, 0.8), loc='upper left', borderaxespad=0, fontsize=20)
for line in leg.get_lines():
    line.set_linewidth(5)
ax.set_title('ROC Curves against Antibody \n(Col1: portal vein)', fontsize=20)
fig.show()

In [ ]:
auroc_lynx = metrics.compute_auroc(
    adata_xenium.obs['t_lynx'].values,
    adata_ab[:, ab_labels].X
)

auroc_gaston = metrics.compute_auroc(
    adata_xenium.obs['t_gaston'].values,
    adata_ab[:, ab_labels].X
)

auroc_coral = metrics.compute_auroc(
    adata_xenium.obs['t_coral'].values,
    adata_ab[:, ab_labels].X
)

auroc_pca = metrics.compute_auroc(
    adata_xenium.obs['t_pca'].values,
    adata_ab[:, ab_labels].X
)

auroc_scvi = metrics.compute_auroc(
    adata_xenium.obs['t_scvi'].values,
    adata_ab[:, ab_labels].X
)

auroc_diffmap = metrics.compute_auroc(
    adata_xenium.obs['t_diffmap'].values,
    adata_ab[:, ab_labels].X
)

auroc_dpt = metrics.compute_auroc(
    adata_xenium.obs['t_dpt'].values,
    adata_ab[:, ab_labels].X
)


In [ ]:
# ts = [t_scvi, t_multivi, t_clue, t_mofa, t_lynx]

# zones = [zone_scvi, zone_multivi, zone_clue, zone_mofa, zone_lynx]

rocs = np.array([
    auroc_pca,
    auroc_diffmap,
    auroc_dpt,
    auroc_scvi,
    auroc_coral,
    auroc_gaston, 
    auroc_lynx
])

methods = ['PCA', 'Diffmap', 'DPT', 'scVI', 'CORAL', 'GASTON', 'LYNX']

In [ ]:
# AUROC score against thresholded antibodies

thresholds = np.linspace(0.1, 0.9, 9)
n_thresholds = auroc_lynx.shape[0]
n_antibodies = auroc_lynx.shape[1]
n_methods = len(methods)
n_repeats = auroc_scvi.shape[0]*auroc_scvi.shape[1]

plot_df = pd.DataFrame({
    'AUROC': rocs.flatten(),
    'Methods': np.repeat(methods, n_repeats),
    'threshold': np.tile(np.repeat(thresholds, n_antibodies), n_methods).flatten()
})

custom_palette = [
    "#003f5c",  # Deep Blue
    "#ffa600",  # Bright Orange
    "#bc5090",  # Vivid Magenta
    "#ff6361",  # Bright Red
    "#9e3b1e",  # Coffee?
    "#28a745",  # Vivid Green
    "#d400ff"   # Bright Purple 
]

fig, ax = plt.subplots(figsize=(5, 3.5), dpi=300)
sns.lineplot(data=plot_df, x='threshold', y='AUROC', hue='Methods', palette=custom_palette,
             marker='o', markersize=5, legend='full',
             errorbar=('se', .5), ax=ax)
ax.set_title('AUROC vs. Antibody Validation', y=1.05, fontsize=12)
ax.spines[['right', 'top']].set_visible(False)
# ax.legend(loc='right', fontsize=8, bbox_to_anchor=(1.0, 0.9))
ax.set_xlabel('Antibody threshold')
ax.set_ylim([0.3, 1])

plt.legend(bbox_to_anchor=(1.05, 0.8), loc='upper left', borderaxespad=0)
plt.show()

del thresholds, n_thresholds, n_antibodies, n_methods, n_repeats

In [ ]:
fig.savefig('../results/plots/auroc.pdf', bbox_inches = "tight", dpi=300)

In [ ]:
# Fitted Antibody expressions in zones
import statsmodels.api as sm
from scipy.stats import zscore, gaussian_kde
from scipy.stats import wasserstein_distance
from sklearn.preprocessing import QuantileTransformer

def disp_antibody_kdes(
    zonation, 
    adata_antibody,
    ncols=4,
    title=None,
    show_plot=True,
    return_density=False
):
    """
    Display KDE estimates of antibody distributions
    in discretized zonation assignments
    """
    markers = adata_antibody.var_names
    factors = np.unique(zonation)

    qt = QuantileTransformer()
    adata_ab = adata_antibody.copy()
    for marker in markers:
        expr = adata_antibody[:,  marker].X
        adata_ab[:, marker].X = qt.fit_transform(
            expr if isinstance(expr, np.ndarray) else
            expr.A
        )

    nrows = len(factors) // ncols
    if len(factors) % ncols != 0:
        nrows += 1

    densities = {}
    
    idx = 0
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.2*nrows))
    for r in range(nrows):
        for c in range(ncols):
            if idx >= len(factors):
                axes[r, c].axis('off')
                continue
                
            for marker in markers:                
                xx = adata_ab[zonation == factors[idx],  marker].X.squeeze()
                sns.kdeplot(xx, alpha=0.2, fill=True, ax=axes[r, c], label=marker)

                kde = sm.nonparametric.KDEUnivariate(xx).fit()
                densities.setdefault(marker, []).append((kde.support, kde.density))
                
            axes[r, c].set_title('Factor {}'.format(factors[idx]), fontsize=20)
            axes[r, c].set_xlabel('Expression', fontsize=15)
            axes[r, c].set_ylabel('Density', fontsize=15)
            axes[r, c].spines[['right', 'top']].set_visible(False)

            if idx == len(factors)-1:
                axes[r, c].legend(loc='right', bbox_to_anchor=(1.3, 0.5))
            idx += 1

    plt.tight_layout()
    plt.suptitle(title, y=1.05, fontsize=35)
    plt.show()

    if return_density:
        return densities
    return None


def get_antibody_distances(densities):
    """
    Compute pairwise distance of fitted antibody KDE peaks 
    along the discretized zonations
    """

    def _pairwise_wasserstain(values):
        values = np.array(values)
        distance = []
        for i in range(values.shape[0]-1):
            for j in range(i+1, values.shape[0]):
                distance.append(wasserstein_distance(values[i], values[j]))
        return distance
    
    antibody_labels = list(densities.keys())
    n_zones = len(densities[antibody_labels[0]])

    distances = []
    for i in range(n_zones):
        zone_densities = np.array([
            densities[label][i][1]
            for label in antibody_labels
        ])
        distances.append(_pairwise_wasserstain(zone_densities))
    return distances


In [ ]:
kde_distances = []
for zone, method in zip(zones, methods):
    densities = disp_antibody_kdes(zone, adata_ab, title=method, return_density=True)
    kde_distance = get_antibody_distances(densities)
    kde_distances.append(kde_distance)
del zone, method, kde_distance

In [ ]:
# TMP

sq.gr.spatial_neighbors(adata_xenium, coord_type='generic', delaunay=True)

sq.gr.spatial_autocorr(
    adata_xenium, attr='obs', genes=['t_pca'],
    mode='moran', transformation=False
)
moran_pca = adata_xenium.uns['moranI'].loc['t_pca', 'I']

sq.gr.spatial_autocorr(
    adata_xenium, attr='obs', genes=['t_diffmap'],
    mode='moran', transformation=False
)
moran_diffmap = adata_xenium.uns['moranI'].loc['t_diffmap', 'I']

sq.gr.spatial_autocorr(
    adata_xenium, attr='obs', genes=['t_dpt'],
    mode='moran', transformation=False
)
moran_dpt = adata_xenium.uns['moranI'].loc['t_dpt', 'I']

sq.gr.spatial_autocorr(
    adata_xenium, attr='obs', genes=['t_scvi'],
    mode='moran', transformation=False
)
moran_scvi = adata_xenium.uns['moranI'].loc['t_scvi', 'I']

sq.gr.spatial_autocorr(
    adata_xenium, attr='obs', genes=['t_coral'],
    mode='moran', transformation=False
)
moran_coral = adata_xenium.uns['moranI'].loc['t_coral', 'I']

sq.gr.spatial_autocorr(
    adata_xenium, attr='obs', genes=['t_gaston'],
    mode='moran', transformation=False
)
moran_gaston = adata_xenium.uns['moranI'].loc['t_gaston', 'I']

sq.gr.spatial_autocorr(
    adata_xenium, attr='obs', genes=['t_lynx'],
    mode='moran', transformation=False
)
moran_lynx = adata_xenium.uns['moranI'].loc['t_lynx', 'I']

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
labels = methods
#counts = [moran_scvi, moran_multivi, moran_clue, moran_mofa, moran_lynx]
#colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']



morans = np.array([
    moran_pca,
    moran_diffmap,
    moran_dpt,
    moran_scvi,
    moran_coral,
    moran_gaston,
    moran_lynx
])


custom_palette = [
    "#003f5c",  # Deep Blue
    "#ffa600",  # Bright Orange
    "#bc5090",  # Vivid Magenta
    "#ff6361",  # Bright Red
    "#9e3b1e",  # Coffee?
    "#28a745",  # Vivid Green
    "#d400ff"   # Bright Purple 
]

ax.bar(labels, morans, label=labels, color=custom_palette)
ax.set_xlabel('Method')
ax.set_ylabel(r"$I$")
ax.text(3, -.2, r"$higher\ is\ better$", ha='center', fontsize=10)
ax.set_title("Moran's I", fontsize=15)
ax.spines[['right', 'top']].set_visible(False)
plt.show()

del labels, morans

In [ ]:
# Centrality
n_nodes = len(centrality_scvi)
plot_df = pd.DataFrame({
    'Degree': list(centrality_scvi) + list(centrality_multivi) + list(centrality_clue) + \
              list(centrality_mofa) + list(centrality_lynx),
    'Method': ['scVI']*n_nodes + ['MultiVI']*n_nodes + ['CLUE']*n_nodes + \
              ['MOFA+']*n_nodes + ['LYNX']*n_nodes
})

fig, ax = plt.subplots(figsize=(6, 4))
ax = sns.boxplot(data=plot_df, x='Method', y='Degree', hue='Method', 
                 #errorbar=("ci", 95), 
                 #err_kws={"color": ".25", "linewidth": 1},
                 ax=ax)

ax.set_title('Degree of Centrality', fontsize=15)
ax.text(2, -.16, r"$lower\ is\ better$", ha='center', fontsize=8)
ax.spines[['right', 'top']].set_visible(False)
plt.show()

del n_nodes, plot_df

In [ ]:
disentangles = []
for i in range(len(kde_distances)):
    disentangles.append(np.sum(kde_distances[i]))

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

labels = methods
counts = disentangles
colors = ['tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple']

ax.bar(labels, counts, label=labels, color=colors)
ax.set_xlabel('Method')
ax.set_ylabel('Total Wasserstein distance')
ax.set_title('Disentanglement Measure:\n (Total antibody-marker distances \n along the spatial gradients)', fontsize=12)
ax.text(2, -4.5, r"$higher\ is\ better$", ha='center', fontsize=8)
ax.spines[['right', 'top']].set_visible(False)
plt.show()

del labels, counts, colors

---